# The names experiment: AquarIum

Each model got AquarIum's brief with the building part taken out (see `aquarium/prompt.md`), so it only imagines its creature and sends back a **name** and a **description**, 50 times, at its default temperature, with the same thinking effort and host as its gallery piece. Answers are written by `npm run names` to `experiments/names/aquarium/<model>/<n>.json`.

How much does each model repeat itself, and how much do the models repeat each other?

**How this differs from the gallery.** In the gallery, each model built its piece (a whole HTML page, drawn in code) and named it. Here it doesn't build anything, and the prompt is slightly different: the technical section is gone, and it's asked to just imagine the piece and send back its name and description. Naming might go a little differently when a model is also planning a drawing. Still, the creative brief and the naming instructions are word for word the same, and the name is exactly what we're studying, so this is a good proxy for the gallery's naming: cheap enough to ask many times over, which the gallery never could be.

**Unlike the planetarium's, this brief steers the names**: it asks for "alliteration or onomatopoeia" and gives three examples ("Jumping Jellyfish", "Screaming Salamander", "Blubbering Blobfish"). So expect lots of alliteration, and watch for echoes of the examples; there's a section on each near the end.

The shared code is in `names.py`; it uses only pandas, so the "charts" are shaded tables.

In [1]:
from collections import Counter
import pandas as pd
from names import *

pd.set_option('display.max_colwidth', 80)
pd.set_option('display.max_rows', 200)

EXHIBIT = 'aquarium'
raw, df, failed = load(EXHIBIT)
models = models_by_answers(df)

## The headlines

In [2]:
rep = repeats(df)
within, across = within_and_across(df)
top_name, top_n = Counter(df.name).most_common(1)[0]
most, least = rep['distinct names'].idxmax(), rep['distinct names'].idxmin()
print(f'{len(df)} answers from {df.model.nunique()} models ({len(failed)} without a name)')
print(f'{df.name.nunique()} distinct names; the most common, "{top_name}", came up {top_n} times')
print(f'word overlap between two names: {within:.2f} from the same model, {across:.2f} from different models')
print(f'most varied: {most} ({rep.loc[most, "distinct names"]} distinct names in {rep.loc[most, "answers"]})')
print(f'least varied: {least} ({rep.loc[least, "distinct names"]} distinct names in {rep.loc[least, "answers"]})')

600 answers from 12 models (0 without a name)
371 distinct names; the most common, "whispering whelk waltzer", came up 18 times
word overlap between two names: 0.11 from the same model, 0.02 from different models
most varied: Mistral Large 4 (49 distinct names in 50)
least varied: Claude Sonnet 5.5 (19 distinct names in 50)


## Clean

- Answers with no name are set aside (counted, not analyzed).
- **Name**: lowercase, no leading "the", hyphens and apostrophes folded, so "The Lantern-Owl" and "lantern owl" count as the same name.
- **Words**: the name's words minus filler (the, of, a…).
- **Figure**: the thing the name is about: its last word, or the last word before "of", "who" or "that" ("The Lantern **Moth**", "The **Kite** of Questions"). A rough heuristic, but a useful one.

In [3]:
df[['model', 'run', 'title', 'name', 'figure']].head(12)

,model,run,title,name,figure
0,Claude Opus 5.5,1,Noodling Nudibranch,noodling nudibranch,nudibranch
1,Claude Opus 5.5,10,Curious Cursive Cuttlefish,curious cursive cuttlefish,cuttlefish
2,Claude Opus 5.5,11,Cogitating Cuttlefish,cogitating cuttlefish,cuttlefish
3,Claude Opus 5.5,12,Pondering Paper Nautilus,pondering paper nautilus,nautilus
4,Claude Opus 5.5,13,Cursive Cuttlefish,cursive cuttlefish,cuttlefish
5,Claude Opus 5.5,14,Curious Calligraphy Cuttlefish,curious calligraphy cuttlefish,cuttlefish
6,Claude Opus 5.5,15,Scribbling Squidlet,scribbling squidlet,squidlet
7,Claude Opus 5.5,16,Curious Calligraphy Cuttlefish,curious calligraphy cuttlefish,cuttlefish
8,Claude Opus 5.5,17,Doodling Dumbo Octopus,doodling dumbo octopus,octopus
9,Claude Opus 5.5,18,Doodling Dumbo Octopus,doodling dumbo octopus,octopus


## Overview: answers, failures, tokens and cost per model

In [4]:
ov = overview(raw)
print(f'total cost: ${ov.cost_total.sum():.2f}')
ov

total cost: $5.68


,answers,no_name,out_tokens_avg,seconds_avg,cost_total
model,,,,,
Muse Wonder,50,0,1133,10.2,1.1624
Qwen3.8 Max Prime,50,0,1273,14.8,0.8355
Gemini 4 Hydrogen,50,0,1936,10.5,0.7533
Claude Opus 5.5,50,0,568,7.8,0.6583
Gemini 4 Carbon,50,0,1197,20.0,0.6351
Mistral Large 4,50,0,5929,96.8,0.6282
Grok 4.7,50,0,1008,20.3,0.3469
Kimi K3.1 Pro,50,0,444,19.3,0.2406
GPT-6.1 Sol,50,0,370,12.5,0.2182


## Every name, by model

In [5]:
every_name(df)

model,Claude Opus 5.5,Claude Sonnet 5.5,GLM 5.3 Prime,GPT-6 Luna,GPT-6.1 Sol,Gemini 4 Carbon,Gemini 4 Hydrogen,Grok 4.7,Kimi K3.1 Pro,Mistral Large 4,Muse Wonder,Qwen3.8 Max Prime
1,Noodling Nudibranch,Pondering Pompom Pufferfish,Murmuring Moray Moppet,Purring Pebblefish,Pitter-Patter Pleatfin,Clinking Curio Cuttlefish,Clickety Cuttlefish,Pondering Pipefish,Pondering Pebblepus,Pulsing Poet Cuttlefish,Pondering Pocket Octopus,Pondering Puffgudge
2,Curious Cursive Cuttlefish,Whispering Whelk Wizard,Pouting Pufferfish,Purring Pebble Loach,Puttering Pebble Puffer,Click-Clack Cuttlefish,Puttering Pattern Puffer,Murmuring Mandarinfish,Murmuring Moonsnail,Bubble-Blowing Betta,Scribbling Sequin Seahorse,Baffled Bobtail
3,Hmm-Hmm Nautilus,Pondering Pearl Pufferfish,Pondering Paddlefin,Blooping Bellfish,Puttering Pebblepuff,Squiggling Scribble Squid,Cataloging Cuttlefish,Humming Hermit Crab,Blooping Bibliophish,Scribbling Stardust Squid,Doodling Daydream Octopus,The Pondering Puffglow
4,Pondering Pebble Puffer,Whispering Whorl Whelk,Wandering Wobblepuss,Purring Pebblefish,Puttering Puzzle Cuttlefish,Click-Clack Cuttlefish,Scribbling Squid,Pondering Pipefish,Blinking Bobtail,Scribbling Sighing Snail,Blooping Bookish Octopus,Flickering Footnote
5,Cartographer Cuttlefish,Tinkering Tidepool Tinselfish,Wandering Wobblefin,Murmuring Moonmoth,Puttering Prismfish,Clickety Curio Cuttlefish,Puttering Pattern-Puffer,Chattering Cuttlefish,Bumbling Bubble Bobtail,Blubbering Button Puffer,Wondrous Wobbling Wordfish,Waffling Wobble-Cuttle
6,Curious Cartographer Cuttlefish,Whispering Whelk Waltz,Waltzing Warthog Nudibranch,Bumbling Bubble Blenny,Puttering Paperfin,Clickety-Clack Cuttlefish,Puttering Pebble-Puffer,Purring Pipefish,Pondering Pocket Puffer,Puffing Picasso Puffer,Doodling Dumbo Octopus,Plinking Plume
7,Noodling Nudibranch,Lantern-Lipped Lollopfish,Murmuring Moontail,Purring Pebblefish,Puttering Pebblepuff,Squiggling Scribble Squid,Shimmering Scribble-Squid,Glimmering Ghost Shrimp,Lunar Loiterfish,Pebble Picasso Puffer,Bubbling Bookish Betta,Muttering Mollusk
8,Murmuring Mosaic Cuttle,Whispering Whelk Wizard,Murmuring Mirrorfin,Murmuring Moonmoth,Puttering Pleatfin,Squiggling Scribble Squid,Color-Coding Cuttlefish,Murmuring Mosaic Cuttle,Pondering Pebble Octopus,Bouncing Blenny,Scribbling Sequin Seahorse,Sputtering Scribblefish
9,Noodling Nudibranch,Whispering Whelk Waltzer,Pondering Puddlefin,Murmuring Mothfish,Puttering Pleatfin,Clinking Curio-Cuttle,Blooping Bobtail Squid,Pondering Pipefish,Murmuring Moonjelly,Pouting Pearl Peeper,Scribbling Starry Squid,Whispering Wobblegill
10,Curious Cursive Cuttlefish,Whispering Whelk Wizard,Murmuring Moray Eel,Murmuring Mothfish,Puttering Pebblepuff,Click-Clack Cuttlefish,Shimmering Scribble Squid,Tip-Tap Ghost Shrimp,Blooping Bubble Snail,Puffing Pattern Puffer,Blooping Bookish Betta,Puttering Pebble-Puff


## How often does each model repeat itself?

- **distinct names**: how many different names (after cleaning) out of its answers.
- **top name**: its most common name and how many times it gave it.
- **distinct figures**: how many different things its names are about.
- **figure entropy**: spread of figures in bits; 0 means always the same figure.
- **name overlap**: average word overlap (Jaccard) between two of its names; 0 = no shared words, 1 = identical.

In [6]:
rep.style.bar(subset=['name overlap'], color='#f4a261', vmin=0, vmax=1) \
    .bar(subset=['distinct names', 'distinct figures'], color='#8ecae6', vmin=0, vmax=rep.answers.max())

,answers,distinct names,top name,top name ×,distinct figures,top figure,figure entropy,name overlap
model,,,,,,,,
Claude Sonnet 5.5,50,19,whispering whelk waltzer,18,17,waltzer,3.360000,0.250000
GPT-6.1 Sol,50,27,puttering pebblepus,7,18,cuttlefish,3.420000,0.230000
Gemini 4 Carbon,50,25,click clack cuttlefish,12,10,cuttlefish,2.020000,0.170000
Claude Opus 5.5,50,29,curious cursive cuttlefish,8,9,cuttlefish,2.130000,0.140000
Grok 4.7,50,27,pondering pipefish,15,13,pipefish,2.820000,0.140000
Gemini 4 Hydrogen,50,22,noodling nautilus,6,8,cuttlefish,2.520000,0.100000
Muse Wonder,50,39,doodling daydream octopus,5,15,octopus,3.270000,0.080000
GPT-6 Luna,50,35,murmuring moonmoth,7,27,pebblefish,4.330000,0.070000
Kimi K3.1 Pro,50,37,murmuring moonfish,4,30,moonfish,4.740000,0.070000


## How much do the models repeat each other?

Word overlap between two answers from **different** models, next to the within-model overlap. If the across-model number is close to the within-model one, the models are drawing from the same well. Below: every name that more than one model came up with.

In [7]:
print(f'average name overlap within a model:  {within:.2f}')
print(f'average name overlap across models:   {across:.2f}')
shared_names(df)

average name overlap within a model:  0.11
average name overlap across models:   0.02


,times,models,by how many models
name,,,
doodling dumbo octopus,10,"Claude Opus 5.5, Gemini 4 Hydrogen, Muse Wonder",3
noodling nautilus,9,"Claude Opus 5.5, Gemini 4 Carbon, Gemini 4 Hydrogen",3
noodling nudibranch,5,"Claude Opus 5.5, Gemini 4 Carbon, Grok 4.7",3
click clack cuttlefish,14,"Gemini 4 Carbon, Gemini 4 Hydrogen",2
puttering pebble puffer,5,"GPT-6.1 Sol, Gemini 4 Hydrogen",2
clickety clack cuttlefish,4,"GPT-6.1 Sol, Gemini 4 Carbon",2
pondering puddlefin,4,"GLM 5.3 Prime, Kimi K3.1 Pro",2
shimmering scribble squid,4,"Gemini 4 Carbon, Gemini 4 Hydrogen",2
chattering cuttlefish,3,"Grok 4.7, Qwen3.8 Max Prime",2


## The words everyone reaches for

Share of answers whose name contains each word, per model and overall (top 25 words).

In [8]:
(word_share(df) * 100).round().astype(int).style.map(lambda v: shade(v, 100)).format('{}%')

,cuttlefish,murmuring,puttering,pondering,puffer,doodling,whispering,whelk,octopus,squid,scribbling,pebble,clack,pipefish,curious,waltzer,dumbo,click,blooping,noodling,blenny,pufferfish,pebblepus,nautilus,scribble
model,,,,,,,,,,,,,,,,,,,,,,,,,
Claude Opus 5.5,58%,10%,0%,6%,2%,10%,0%,0%,12%,0%,8%,2%,0%,0%,32%,0%,10%,0%,0%,8%,0%,0%,0%,6%,4%
Claude Sonnet 5.5,0%,6%,0%,10%,4%,0%,58%,58%,0%,0%,0%,0%,0%,0%,0%,36%,2%,0%,0%,0%,0%,6%,0%,0%,0%
GLM 5.3 Prime,0%,26%,0%,18%,4%,0%,0%,4%,0%,0%,0%,0%,0%,0%,0%,0%,0%,0%,0%,0%,0%,6%,0%,0%,0%
GPT-6 Luna,0%,30%,4%,0%,10%,6%,0%,0%,2%,0%,0%,2%,0%,0%,0%,0%,0%,0%,8%,0%,4%,0%,0%,0%,0%
GPT-6.1 Sol,28%,0%,78%,0%,2%,6%,0%,0%,2%,2%,0%,14%,2%,0%,2%,0%,0%,0%,0%,0%,0%,0%,18%,0%,0%
Gemini 4 Carbon,60%,0%,2%,0%,8%,2%,0%,0%,0%,16%,0%,6%,32%,0%,0%,0%,0%,26%,0%,6%,2%,0%,0%,4%,14%
Gemini 4 Hydrogen,36%,0%,18%,6%,18%,4%,0%,0%,4%,18%,12%,12%,4%,0%,0%,0%,4%,4%,2%,14%,4%,6%,0%,12%,4%
Grok 4.7,4%,16%,2%,32%,0%,0%,0%,0%,0%,0%,0%,0%,2%,40%,2%,0%,0%,2%,0%,2%,6%,0%,0%,0%,0%
Kimi K3.1 Pro,0%,36%,6%,22%,4%,4%,2%,0%,4%,4%,0%,2%,0%,0%,0%,0%,4%,0%,6%,0%,2%,0%,6%,0%,0%


## What the creatures are (the figure), per model

In [9]:
figs = figures(df)
figs.style.map(lambda v: shade(v, figs.drop('ALL MODELS').values.max()))

figure,cuttlefish,puffer,octopus,squid,pipefish,waltzer,blenny,pufferfish,pebblepus,nautilus,shrimp,betta,nudibranch,snail,mandarinfish
model,,,,,,,,,,,,,,,
Claude Opus 5.5,29,1,6,0,0,0,0,0,0,3,0,0,3,2,0
Claude Sonnet 5.5,0,2,0,0,0,18,0,3,0,0,0,0,0,0,0
GLM 5.3 Prime,0,2,0,0,0,0,0,3,0,0,0,0,2,0,0
GPT-6 Luna,0,5,1,0,0,0,2,0,0,0,1,0,0,0,0
GPT-6.1 Sol,14,1,1,1,0,0,0,0,9,0,0,0,0,0,0
Gemini 4 Carbon,30,4,0,8,0,0,1,0,0,2,0,0,1,0,0
Gemini 4 Hydrogen,18,9,2,9,0,0,2,3,0,6,0,0,1,0,0
Grok 4.7,2,0,0,0,20,0,3,0,0,0,6,0,1,0,9
Kimi K3.1 Pro,0,2,2,2,0,0,1,0,3,0,0,0,1,2,0


## The descriptions: most common words

In [10]:
description_words(df)

word,tank,glass,tiny,like,sized,water,whenever,blue,small,pebbles,...,warm,bubbles,glowing,slow,swaying,spirals,skin,constellations,among,quiet
answers,265,230,210,208,205,191,190,177,172,172,...,127,118,117,116,113,109,109,109,103,96
share,44%,38%,35%,35%,34%,32%,32%,30%,29%,29%,...,21%,20%,20%,19%,19%,18%,18%,18%,17%,16%


## Compared with the gallery: each model's actual piece

Did the name a model gave its real gallery piece come up again among its 50 answers here?

In [11]:
vs_gallery(df, EXHIBIT, rep)

,gallery piece,among its answers?,its most common name here
model,,,
Claude Opus 5.5,Doodling Daydream Cuttlefish,False,curious cursive cuttlefish
Claude Sonnet 5.5,Flicker-Flash Cuttlefish,False,whispering whelk waltzer
GLM 5.3 Prime,Murmuring Moonfin,False,ponderous puddlefin
GPT-6 Luna,Opal Octopus,False,murmuring moonmoth
GPT-6.1 Sol,Pleated Puddlewing,False,puttering pebblepus
Gemini 4 Carbon,Pondering Prism-Puffer,False,click clack cuttlefish
Gemini 4 Hydrogen,Noodling Nautilus,True,noodling nautilus
Grok 4.7,Swoop-Swoop Silkfin,False,pondering pipefish
Kimi K3.1 Pro,Murmuring Moonbell,False,murmuring moonfish


## Is 50 answers per model enough?

Distinct names among each model's first *k* answers, as *k* grows. If a model's count stops climbing, more answers won't show much new; if it keeps climbing, the model rarely repeats and more answers would keep finding new names.

In [12]:
growth(df)

,first 3,first 5,first 10,first 20,first 30,first 50
model,,,,,,
Claude Opus 5.5,3,5,7,15,22,29
Claude Sonnet 5.5,3,5,8,11,13,19
GLM 5.3 Prime,3,5,10,20,27,41
GPT-6 Luna,3,4,6,13,22,35
GPT-6.1 Sol,3,5,7,13,18,27
Gemini 4 Carbon,3,4,6,10,15,25
Gemini 4 Hydrogen,3,4,8,16,18,22
Grok 4.7,3,4,8,12,17,27
Kimi K3.1 Pro,3,5,10,17,26,37


## How alliterative are the names?

Share of each model's names in which at least two words start with the same letter ("**B**lubbering **B**lobfish"). The brief asks for alliteration or onomatopoeia, so this shows how literally each model took it.

In [13]:
allit = df.assign(alliterative=df.name.map(alliterative)).groupby('model').alliterative.mean().loc[models]
allit.loc['ALL MODELS'] = df.name.map(alliterative).mean()
(allit * 100).round().astype(int).to_frame('alliterative names').style.bar(color='#8ecae6', vmin=0, vmax=100).format('{}%')

,alliterative names
model,
Claude Opus 5.5,100%
Claude Sonnet 5.5,100%
GLM 5.3 Prime,100%
GPT-6 Luna,100%
GPT-6.1 Sol,100%
Gemini 4 Carbon,100%
Gemini 4 Hydrogen,100%
Grok 4.7,100%
Kimi K3.1 Pro,100%


## Echoes of the examples

How often a name reuses a word from the brief's three examples: jellyfish, salamander, blobfish, jumping, screaming, blubbering.

In [14]:
EXAMPLES = {'jellyfish', 'salamander', 'blobfish', 'jumping', 'screaming', 'blubbering'}
echo = df.words.map(lambda ws: ', '.join(sorted(EXAMPLES & set(ws))))
echoes = df[echo != ''].assign(echo=echo[echo != ''])
print(f'{len(echoes)} of {len(df)} names reuse a word from the examples')
pd.crosstab(echoes.model, echoes.echo) if len(echoes) else 'none'

2 of 600 names reuse a word from the examples


echo,blubbering
model,
Mistral Large 4,2


## Failures

Answers with no name in them (the reply is kept in the `.failed.json` file).

In [15]:
failures(failed)

'none'